## GRID telemetry EDA

Explore the checked-in sample match payload, inspect its normalized structure with Pandas/Seaborn, then export metrics using the backend's deterministic aggregator. Run from the repository root; the sample is illustrative and not official match data.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'backend'))
from app.services.grid_aggregator import aggregate_matches

source_path = project_root / 'data' / 'sample_grid_match.json'
print(f'Project root: {project_root}')
print(f'Sample source: {source_path}')

In [ ]:
payload = json.loads(source_path.read_text(encoding='utf-8'))
matches = payload.get('matches', [])
rows = []
for match in matches:
    side = match.get('focus_side', 'blue')
    stats = match.get('teams', {}).get(side, {})
    rows.append({
        'match_id': match.get('match_id'),
        'focus_side': side,
        'gold_diff_at_15': stats.get('gold_diff_at_15'),
        'first_blood': stats.get('first_blood'),
        'first_tower': stats.get('first_tower'),
        'events': len(match.get('events', [])),
        'jungle_points_0_5': sum(
            1 for path in match.get('jungle_paths', {}).values() for point in path
            if 0 <= float(point.get('minute', 0)) <= 5
        ),
    })
match_df = pd.DataFrame(rows)
print(f'Rows: {len(match_df)}; columns: {list(match_df.columns)}')
print(match_df.dtypes)
print('Missing values:', match_df.isna().sum().to_dict())
display(match_df)

In [ ]:
objective_rows = [
    {'match_id': match.get('match_id'), 'objective': event.get('type'), 'capturing_side': event.get('team')}
    for match in matches for event in match.get('events', [])
]
objective_df = pd.DataFrame(objective_rows)
if not objective_df.empty:
    sns.countplot(data=objective_df, x='objective', hue='capturing_side')
    plt.title('Objective captures in sample matches')
    plt.xlabel('Objective type')
    plt.ylabel('Event count')
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

In [ ]:
metrics = aggregate_matches(matches)
metrics_path = project_root / 'data' / 'scouting_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(f'Aggregated {metrics["matches"]} matches; exported {metrics_path}')
print({key: value for key, value in metrics.items() if key not in ('jungle_pathing_0_5', 'pre_objective_ward_coordinates')})